# 01 · Exploratory data analysis

Goal: understand the data before trusting any model. Run top to bottom after putting the Zindi files in `data/raw/`.

To try it before the real data is available, set `DATA_DIR = ROOT / 'data' / 'synthetic'` after running `python -m src.make_synthetic`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.data import load_dataset
from src.features import build_features

DATA_DIR = ROOT / 'data' / 'raw'
ds = load_dataset(DATA_DIR)
train, test, y = ds.train, ds.test, ds.y
train.head()

## 1. Columns, types and missing values

In [ ]:
summary = pd.DataFrame({
    'dtype': train.dtypes.astype(str),
    'n_unique': train.nunique(),
    'missing_train_%': (train.isna().mean() * 100).round(1),
    'missing_test_%': (test.reindex(columns=train.columns).isna().mean() * 100).round(1),
    'example': train.iloc[0],
})
summary

## 2. Target balance

F1 is computed at a fixed 0.5 threshold. If positives are rare, an unweighted model may predict too few positives; `src.train` checks class weighting automatically.

In [ ]:
print(ds.label_mapping)
y.value_counts(normalize=True).rename('share')

## 3. Numeric features: how do they differ by class?

In [ ]:
fs = build_features(train, test, ds.id_col, ds.target)
num = [c for c in fs.num_cols if not c.endswith('_freq')][:12]
fig, axes = plt.subplots((len(num) + 2) // 3, 3, figsize=(14, 3 * ((len(num) + 2) // 3)))
for ax, col in zip(axes.ravel(), num):
    for label in (0, 1):
        fs.X_train.loc[y == label, col].plot.hist(ax=ax, bins=40, alpha=0.5, density=True, label=f'target={label}')
    ax.set_title(col)
    ax.legend(fontsize=8)
for ax in axes.ravel()[len(num):]:
    ax.axis('off')
plt.tight_layout()

## 4. Categorical features: positive rate per category

In [ ]:
for col in fs.cat_cols:
    rates = (pd.DataFrame({col: fs.X_train[col], 'y': y})
             .groupby(col, observed=True)['y'].agg(positive_rate='mean', count='size')
             .sort_values('count', ascending=False).head(15))
    display(rates.style.format({'positive_rate': '{:.2%}'}).set_caption(col))

## 5. Train vs test drift

If a model can tell train rows from test rows, those features shift between the two sets and CV scores may not match the leaderboard.

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score

both = pd.concat([fs.X_train, fs.X_test], ignore_index=True)
is_test = [0] * len(fs.X_train) + [1] * len(fs.X_test)
adv = LGBMClassifier(n_estimators=200, verbose=-1)
p = cross_val_predict(adv, both, is_test, cv=5, method='predict_proba')[:, 1]
print(f'Adversarial AUC: {roc_auc_score(is_test, p):.3f}  (about 0.5 = no drift)')
adv.fit(both, is_test)
pd.Series(adv.feature_importances_, index=both.columns).sort_values(ascending=False).head(10)